# Lab 03: Parquet Internals, Silver Cleaning Part 1 & Schema Enforcement
**Course:** Business Intelligence and Big Data Analysis (`01 31 0367 00`)  
**Date & Format:** Lab #3 (EN: Mon 23.11.2026 afternoon *following Tutorial #2* / FR: Tue 24.11.2026) · 3 Hours (180 min)

---

## 🎯 1. Context & Architecture-as-Code Lineage
In Lab 02 you staged raw **Case 0 (`EuroPack Beverage Bottling`)** data verbatim into `bronze_staged/*.parquet`. Before performing statistical outlier math and shift integration (Lab 04), today's lab connects **Lecture 02's Apache Parquet internals** directly to **Silver Structural Hygiene & Strict Schema Typing**, aligned with **DAMA Data Quality Rules (`DQR`)** from Tutorial #2:

```mermaid
flowchart TD
    B0["2. Bronze Stage: bronze_staged/*.parquet"] --> P0["3. Parquet Internals & Pushdown (3.1 - 3.4)"]
    P0 --> Q0["4. Read-Only Bronze DQ Profiling & Scorecard (4.1 - 4.3)"]
    Q0 --> S1["5. String Hygiene & Canonical Key Aliasing"]
    S1 --> S2["6. Data Types, Timestamp Parsing, Keep-Last Dedup & S3 Rollover (6.1 - 6.2)"]
    S2 --> P1["7. Enforce Silver Schema Contract -> silver_part1/*.parquet"]
```

**Your Objectives:**
1. **Inspect Apache Parquet Internals (Lecture 02 Bridge):** Verify `PAR1` magic header/footer bytes, decode the footer size, inspect Row Groups, Column Chunks, byte offsets, `RLE_DICTIONARY`/`RLE`/`SNAPPY` compression, footer min/max/null statistics, and **Projection + Predicate Pushdown**.
2. **Profile Bronze (Read-Only):** Compile a **Bronze DAMA DQ Scorecard** without mutating Bronze.
3. **String Hygiene & Aliases:** Resolve whitespace, inconsistent casing (`'Line_A '`, `'line_c'`), legacy aliases (`'LN-B'` -> `'LINE_B'`), and mixed CMMS `stoppage_type` strings (`'unplanned'` -> `'Unplanned'`).
4. **Understand & Enforce Silver Data Types:** Prove why storing timestamps/dates as strings breaks arithmetic, ordering, RAM, and SQL pushdown; parse timestamps (`timestamp[ns]`, `timestamp[ns, tz=UTC]`) and dates (`date32[day]`), deduplicate (`keep='last'`), and fix **Night Shift (`S3`: `22:00–06:00`) midnight crossovers** via `(timestamp - 6h)`.
5. **Persist & Audit Schema Contracts:** Save all 5 typed tables to `silver_part1/*.parquet` and verify embedded Arrow/Parquet schemas and DuckDB `DESCRIBE` types.

---

## ⚙️ 2. Environment Setup & Bronze Input Discovery

### Step 1: Resolve Bronze Inputs, PyArrow/DuckDB Tooling & Silver Part 1 Output Directory

In [ ]:
from pathlib import Path

_cwd = Path.cwd().resolve()
REPO_ROOT = None
for _cand in [_cwd, *_cwd.parents]:
    if (_cand / "03_labs_student_starters").exists():
        REPO_ROOT = _cand
        break
    for _sub in ("ife_bi_bigdata_course", "ife_bi_bigdata_student"):
        if (_cand / _sub / "03_labs_student_starters").exists():
            REPO_ROOT = _cand / _sub
            break
    if REPO_ROOT is not None:
        break
assert REPO_ROOT is not None, f"Could not locate 03_labs_student_starters from {_cwd}"

CASE_NAME = "case0_europack_bottling"
STUDENT_BRONZE_DIR = REPO_ROOT / "03_labs_student_starters" / "lab_02_bronze_ingestion_and_profiling" / "data" / CASE_NAME
REF_CASE_DATA = REPO_ROOT / "05_case_studies_reference_solutions" / CASE_NAME / "data"
BRONZE_RAW = STUDENT_BRONZE_DIR if STUDENT_BRONZE_DIR.exists() else (REF_CASE_DATA / "bronze")
SANDBOX_DATA = REPO_ROOT / "03_labs_student_starters" / "sandbox_outputs" / CASE_NAME


def resolve_stage_dir(stage_name: str) -> Path:
    """Read upstream stage from student sandbox_outputs if populated, else fallback read-only to reference solution."""
    cand = SANDBOX_DATA / stage_name
    if cand.exists() and any(cand.glob("*.parquet")):
        return cand
    ref_cand = REF_CASE_DATA / stage_name
    if ref_cand.exists():
        return ref_cand
    raise FileNotFoundError(
        f"Stage '{stage_name}' not found in {cand} (and no reference fallback exists). Run the prerequisite Lab notebook first."
    )
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

BRONZE_STAGED = resolve_stage_dir("bronze_staged")
SILVER_P1 = SANDBOX_DATA / "silver_part1"
SILVER_P1.mkdir(parents=True, exist_ok=True)

df_telem = pd.read_parquet(BRONZE_STAGED / "bronze_telemetry.parquet")
df_stop = pd.read_parquet(BRONZE_STAGED / "bronze_stoppages.parquet")
df_lines = pd.read_parquet(BRONZE_STAGED / "bronze_lines.parquet")
df_shifts = pd.read_parquet(BRONZE_STAGED / "bronze_shifts.parquet")
df_formats = pd.read_parquet(BRONZE_STAGED / "bronze_bottle_formats.parquet")

print("Bronze source :", BRONZE_STAGED)
print("Silver Part 1 :", SILVER_P1)
print(f"Loaded telemetry: {len(df_telem):,} rows | Stoppage logs: {len(df_stop):,} rows")


## 📦 3. Inside Apache Parquet — Anatomy, Header, Footer & Pushdown (Lecture 02 Bridge)
In **Lecture 02 (§5.3)** you saw why Apache Parquet beats CSV/JSON for analytical storage. Let's open `bronze_telemetry.parquet` as raw binary bytes and via `pyarrow.parquet.ParquetFile`—**without loading the data rows**—to verify every part of its internal layout:

```text
 [0 .. 4)              Magic Header: b'PAR1' (4 bytes)
 [4 .. footer_start)   Row Groups -> Column Chunks -> Pages (Dictionary + RLE + Snappy)
 [footer_start .. -8)  File Footer Metadata (Thrift: schema, byte offsets, min/max/null stats)
 [-8 .. -4)            4-byte little-endian integer = Footer length in bytes
 [-4 .. EOF)           Magic Footer: b'PAR1' (4 bytes)
```

### 3.1 Demo: Magic Bytes (`PAR1`), Footer Length, Metadata & Embedded Schema
**Why is the File Footer placed at the end and read FIRST?** Because while writing, the engine does not know the final byte offsets or min/max statistics until the last row is flushed. When reading, Parquet jumps directly to `EOF - 8`, reads the 4-byte `footer_len`, and loads the File Footer metadata before touching a single data page!

*Run the ready-to-launch demo cell below to inspect the raw bytes and embedded Arrow schema of `bronze_telemetry.parquet`.*

In [ ]:
# Ready-to-run Demo: Inspect the raw binary structure and metadata footer of a Parquet file
pq_path = BRONZE_STAGED / "bronze_telemetry.parquet"

# Read the entire file as raw bytes so we can verify its binary header/footer layout
raw_bytes = pq_path.read_bytes()

# 1. Verify the 4-byte "PAR1" magic signature at the very start and very end of the file
magic_header = raw_bytes[:4]
magic_footer = raw_bytes[-4:]

# 2. The 4 bytes right before the final "PAR1" (indices -8..-4) store the Footer length
#    as a little-endian integer (least-significant byte first, per the Parquet spec)
footer_len = int.from_bytes(raw_bytes[-8:-4], "little")

# 3. Open ONLY the Parquet metadata handle — this reads the Footer, NOT the row data pages
pf = pq.ParquetFile(pq_path)

# Sanity checks: confirm valid PAR1 markers and that our decoded footer_len matches PyArrow
assert magic_header == b"PAR1" and magic_footer == b"PAR1", "Invalid Parquet magic bytes!"
assert footer_len == pf.metadata.serialized_size, "Footer length mismatch!"

print(f"File size         : {len(raw_bytes):,} bytes")
print(f"Magic header/tail : {magic_header} ... {magic_footer}")
print(f"Footer length     : {footer_len:,} bytes (matches pf.metadata.serialized_size)")
print(f"Rows / Cols / RGs : {pf.metadata.num_rows:,} rows | {pf.metadata.num_columns} cols | {pf.metadata.num_row_groups} Row Group(s)")
print(f"Format / Writer   : v{pf.metadata.format_version} | {pf.metadata.created_by}")
print(f"Custom KV keys    : {list((pf.metadata.metadata or {}).keys())}")
print("\nEmbedded Arrow Schema (notice 'timestamp' & '_ingestion_timestamp_utc' are still STRING in Bronze!):")
print(pf.schema_arrow)


### 3.2 Demo: Column Chunks, Byte Offsets, Encodings (`Dictionary` + `RLE` + `Snappy`) & Footer Statistics
Run this demo to inspect `pf.metadata.row_group(0)` column-by-column **strictly from the metadata footer** (without reading or decompressing any data pages) and observe:
1. **Columnar Byte Layout:** Each Column Chunk starts right after the previous chunk (`dictionary_page_offset` / `data_page_offset`), and `4 + sum(compressed_B) + footer_len + 8` equals the **exact byte size of the file**!
2. **Dictionary + RLE Compression:** Look at `_source_file` and `_ingestion_timestamp_utc`: `23,164` repeated strings compress down to ~`200–265 bytes`!
3. **Free DQ Diagnostics from Footer Statistics:** Look at `nulls`, `min`, and `max`—we can already spot `120` missing temperatures, `-99.0`/`85.0 bar` filler pressure, `-120` bottled units, `'LINE_A'`..`'line_c'`, and `'2099-01-01'` *without decompressing a single row*!

In [ ]:
# Ready-to-run Demo: Read per-column layout & min/max/null stats directly from Footer metadata
rg = pf.metadata.row_group(0)

chunk_rows = []
for i in range(rg.num_columns):
    # c = metadata descriptor for column i; st = precomputed min/max/null_count stats
    c = rg.column(i)
    st = c.statistics
    chunk_rows.append({
        "column": c.path_in_schema,                 # Column name in the file schema
        "phys_type": c.physical_type,               # Low-level Parquet storage type (INT64, DOUBLE, BYTE_ARRAY)
        "comp": c.compression,                      # Compression codec (SNAPPY)
        "encodings": ",".join(c.encodings),         # Encodings used (Dictionary + RLE + PLAIN)
        "dict_off": c.dictionary_page_offset,       # Byte offset where the dictionary page starts
        "data_off": c.data_page_offset,             # Byte offset where the compressed data pages start
        "raw_B": c.total_uncompressed_size,         # Uncompressed size of this column chunk (bytes)
        "comp_B": c.total_compressed_size,          # Compressed size on disk (bytes)
        "nulls": st.null_count if st else None,     # Missing values count stored in footer
        "min_stat": repr(st.min) if st and st.has_min_max else None,  # Min value in Row Group
        "max_stat": repr(st.max) if st and st.has_min_max else None,  # Max value in Row Group
    })

df_chunks = pd.DataFrame(chunk_rows)

# Verify the exact binary byte budget:
# 4B header ("PAR1") + all compressed column chunks + Thrift footer + 8B tail (4B len + 4B "PAR1")
total_payload_b = df_chunks["comp_B"].sum()
assert 4 + total_payload_b + footer_len + 8 == len(raw_bytes), "Byte map mismatch!"

print(f"Exact binary map: 4B (PAR1) + {total_payload_b:,}B (Column Chunks) + {footer_len:,}B (Footer) + 8B (len+PAR1) = {len(raw_bytes):,}B")
print("\nColumn Chunk Metadata & Footer Statistics (Row Group 0):")
print(df_chunks.to_string(index=False))


### 3.3 Demo: Projection Pushdown (`columns=[...]`)
Reading only `["line_id", "filler_pressure_bar"]` uses the byte offsets in the Parquet footer to jump directly to those 2 columns and skip >80% of Parquet column chunk bytes (and >96% compared to scanning the raw CSV!).

In [ ]:
# Ready-to-run Demo: Projection Pushdown (skip unused columns using Footer byte offsets)
selected_cols = ["line_id", "filler_pressure_bar"]
csv_bytes = (BRONZE_RAW / "line_telemetry.csv").stat().st_size if (BRONZE_RAW / "line_telemetry.csv").exists() else 0

# PyArrow reads ONLY the 2 requested columns instead of all 12
tbl_proj = pq.read_table(pq_path, columns=selected_cols)

# Sum the compressed byte sizes of only those 2 Column Chunks from Row Group 0 metadata
proj_bytes = sum(
    rg.column(i).total_compressed_size
    for i in range(rg.num_columns)
    if rg.column(i).path_in_schema in selected_cols
)
print(f"Projection Pushdown ({selected_cols}):")
print(f"  Raw CSV scan: {csv_bytes:,} B -> Full Parquet chunks: {total_payload_b:,} B -> Projected 2 cols: {proj_bytes:,} B ({100 * (1 - proj_bytes / total_payload_b):.1f}% Parquet I/O skipped!)")
print(f"  Projected table shape: {tbl_proj.num_rows:,} rows x {tbl_proj.num_columns} cols")


### 3.4 Demo: Multi-Row-Group Predicate Pushdown (`filters=[...]`)
When a Parquet file spans multiple Row Groups (`row_group_size=6000`), the query engine inspects each Row Group's footer `[min, max]` and **skips entire Row Groups** whose range cannot match! Notice also how uncleaned `'line_c'` lands in Row Group 3 (`'l' > 'L'` in ASCII), proving why Section 5 string hygiene matters for filtering.

In [ ]:
# Ready-to-run Demo: Multi-Row-Group Predicate Pushdown (skip whole Row Groups using [min, max] stats)
# Write an in-memory Parquet buffer sorted by line_id and split into 4 Row Groups
sink = pa.BufferOutputStream()
pq.write_table(
    pa.Table.from_pandas(df_telem.sort_values("line_id"), preserve_index=False),
    sink,
    row_group_size=6000,   # Splits 23,164 rows into 4 Row Groups (~6,000 rows each)
    compression="snappy",
)
multi_buf = sink.getvalue()
pf_multi = pq.ParquetFile(pa.BufferReader(multi_buf))

# Apply a filter pushdown: PyArrow checks each Row Group's [min, max] footer stats BEFORE reading data
tbl_pred = pq.read_table(
    pa.BufferReader(multi_buf),
    columns=["timestamp", "line_id", "filler_pressure_bar"],
    filters=[("line_id", "=", "LINE_C")],
)

print(f"Multi-Row-Group Stats ({pf_multi.metadata.num_row_groups} Row Groups, sorted by line_id):")
for r in range(pf_multi.metadata.num_row_groups):
    rg_r = pf_multi.metadata.row_group(r)
    st_m = rg_r.column(1).statistics  # column 1 is line_id
    # If "LINE_C" falls outside [min, max], the entire Row Group is skipped without decompression
    can_skip = not (st_m.min <= "LINE_C" <= st_m.max)
    print(f"  RG {r}: rows={rg_r.num_rows:>4} | line_id=[{st_m.min!r}, {st_m.max!r}] -> {'SKIPPED by footer!' if can_skip else 'CANDIDATE'}")
print(f"Predicate Pushdown result ('LINE_C'): {tbl_pred.num_rows:,} rows (notice uncleaned 'line_c' in RG 3 was missed!)")


## 🔍 4. Initial Bronze Data Quality Profiling (Strictly Read-Only!)
**Critical Rule (`Bronze Immutability`):** Now that the Parquet footer already hinted at dirty keys, `120` nulls, extreme pressures, and `2099-*` dates, play forensic detective on the staged Bronze tables (`df_telem`, `df_stop`, `df_lines`) so our **DAMA DQ Rules** catch 100% of anomalies.

### 4.1 DQ Scan: Line Key & Category Casing Drift (`line_id` & `stoppage_type`)
Compare raw keys against the canonical ERP master list (`LINE_A..LINE_D`).

In [ ]:
# Canonical ERP bottling line set (e.g. {'LINE_A', 'LINE_B', 'LINE_C', 'LINE_D'})
valid_lines = set(df_lines["line_id"].astype(str).str.strip().str.upper())

# TODO: Inspect raw telemetry `line_id` counts (dropna=False) and CMMS `stoppage_type` counts (dropna=False)
raw_line_counts = None     # Compute value_counts(dropna=False) on df_telem["line_id"]
stop_type_counts = None    # Compute value_counts(dropna=False) on df_stop["stoppage_type"]

if raw_line_counts is not None and stop_type_counts is not None:
    print("Canonical ERP lines:", sorted(valid_lines))
    print("\n1. Raw PLC line_id strings (notice 'Line_A ', 'LN-B', 'line_c' & orphan 'LINE_Z'):")
    for k, cnt in raw_line_counts.items():
        print(f"  {repr(k):<14} -> {cnt:>6,} rows")
    print("\n2. Stoppage log stoppage_type casing drift ('Unplanned' vs 'unplanned' vs 'Preventive'):")
    print(stop_type_counts.to_string())
else:
    print("⏳ [TODO] Assign raw_line_counts and stop_type_counts above.")


### 4.2 DQ Scan: Physics Bounds, Nulls (`NaN`), Future Dates & Duplicates
Apply vectorized boolean masks (`<`, `>`, `|`, `~`, `.isin()`) to diagnose:
1. Impossible rotary filler pressure (`< 0.0` like `-99.0 bar` or `> 12.0` like `85.0 bar`).
2. Missing pasteurizer temperature readings (`NaN` in `pasteurizer_temp_C`) — remember: **never `dropna()`**, or you delete valid bottled units!
3. `rejected_units > bottled_units`, future `2099-*` timestamps, duplicate telemetry rows, and orphan/future stoppage logs (`STOP-20269999`).

In [ ]:
# TODO: Write vectorized Pandas expressions to compute each diagnostic counter below:
#   - bad_pressure_count : count rows where filler_pressure_bar < 0 or filler_pressure_bar > 12
#   - null_temp_count    : count missing (NaN) values in pasteurizer_temp_C
#   - dupes_count        : count duplicate rows on composite key ["timestamp", "line_id", "bottle_format"]
bad_pressure_count = None
null_temp_count = None
dupes_count = None

tel_ts = pd.to_datetime(df_telem["timestamp"], errors="coerce")
future_tel_count = (tel_ts > "2027-01-01").sum()
bad_reject_count = (df_telem["rejected_units"] > df_telem["bottled_units"]).sum()

# Step-by-step Stoppage Log DQ checks (broken into readable boolean masks):
# 1. Normalize line_code to check whether it matches a known ERP line (catches 'LN-B', 'LINE_Z', etc.)
clean_stop_codes = df_stop["line_code"].astype(str).str.strip().str.upper()
is_unknown_line = ~clean_stop_codes.isin(valid_lines)

# 2. Parse event_date and flag unparseable (NaT) or impossible future (> 2027) dates
stop_ts = pd.to_datetime(df_stop["event_date"], errors="coerce")
is_invalid_date = stop_ts.isna()
is_future_date = stop_ts > "2027-01-01"

# 3. Combine the three masks with bitwise OR ('|') so any problematic stoppage log is flagged
stop_bad_mask = is_unknown_line | is_invalid_date | is_future_date

if all(v is not None for v in (bad_pressure_count, null_temp_count, dupes_count)):
    print(f"Telemetry out-of-bounds pressure (<0 or >12 bar): {bad_pressure_count}")
    print(f"Telemetry missing pasteurizer_temp_C (NaN)      : {null_temp_count}  (to interpolate in Lab 04, NOT drop!)")
    print(f"Telemetry MQTT duplicate rows                   : {dupes_count}")
    print(f"Telemetry future 2099 timestamps                : {future_tel_count}")
    print(f"Telemetry rejected_units > bottled_units        : {bad_reject_count}")
    print("\nFlagged stoppage logs (non-canonical alias, orphan line, or 2099-* timestamp):")
    display(df_stop.loc[stop_bad_mask, ["log_id", "event_date", "line_code", "stoppage_type", "root_cause", "downtime_min"]].head(8))
    print("  Total stoppage rows flagged above:", stop_bad_mask.sum(), "| Defensive downtime_min <= 0:", (df_stop["downtime_min"] <= 0).sum())
else:
    print("⏳ [TODO] Complete bad_pressure_count, null_temp_count, and dupes_count above.")


### 4.3 Bronze DAMA DQ Scorecard (Triage Plan for Labs 03–04)
Compile your read-only findings into a **`DataFrame` Scorecard** mapping every detected anomaly to its **Lab 03 (Normalize/Deduplicate)** or **Lab 04 (Interpolate/Quarantine)** action.

In [ ]:
if all(v is not None for v in (bad_pressure_count, null_temp_count, dupes_count)):
    orphan_tel_count = (df_telem["line_id"].astype(str).str.strip().str.upper() == "LINE_Z").sum()
    alias_tel_count = df_telem["line_id"].isin(["Line_A ", "LN-B", "line_c"]).sum()
    stop_future_orphan_count = (stop_ts > "2027-01-01").sum()
    stop_unplanned_lower_count = (df_stop["stoppage_type"] == "unplanned").sum()

    dq_scorecard = pd.DataFrame([
        {"Rule_ID": "DQR-KEY", "Source": "PLC + CMMS", "Anomaly": "Whitespace/casing/alias drift ('Line_A ', 'LN-B', 'unplanned')", "Count": alias_tel_count + stop_unplanned_lower_count, "Planned_Action": "Lab 03 Normalize (.strip().upper() + alias map)"},
        {"Rule_ID": "DQR-107", "Source": "PLC CSV", "Anomaly": "MQTT duplicate telemetry rows", "Count": dupes_count, "Planned_Action": "Lab 03 Deduplicate (keep='last')"},
        {"Rule_ID": "DQR-101", "Source": "PLC + CMMS", "Anomaly": "Orphan ghost asset (LINE_Z)", "Count": orphan_tel_count, "Planned_Action": "Lab 04 Quarantine (tier BLOCKER)"},
        {"Rule_ID": "DQR-102", "Source": "PLC + CMMS", "Anomaly": "Future 2099-* timestamps (incl. STOP-20269999)", "Count": future_tel_count + stop_future_orphan_count, "Planned_Action": "Lab 04 Quarantine"},
        {"Rule_ID": "DQR-103", "Source": "PLC CSV", "Anomaly": "Impossible filler_pressure_bar (-99.0 / 85.0 bar)", "Count": bad_pressure_count, "Planned_Action": "Lab 04 Quarantine"},
        {"Rule_ID": "DQR-104", "Source": "PLC CSV", "Anomaly": "rejected_units > bottled_units", "Count": bad_reject_count, "Planned_Action": "Lab 04 Quarantine"},
        {"Rule_ID": "DQR-TEMP", "Source": "PLC CSV", "Anomaly": "Missing pasteurizer_temp_C (NaN)", "Count": null_temp_count, "Planned_Action": "Lab 04 Time Interpolate (NO dropna!)"},
    ])
    print("=== BRONZE DATA QUALITY SCORECARD (TRIAGE PLAN) ===")
    display(dq_scorecard)
else:
    print("⏳ Complete the DQ Scan cells above to generate your Bronze DQ Scorecard.")


## 🧹 5. String Hygiene & Canonical Key Aliasing
Normalize `line_id`, `line_code`, and mixed `stoppage_type` labels deterministically before deduplication.

In [ ]:
# Canonical alias lookup table mapping legacy variants -> standard ERP bottling line IDs
LINE_ALIASES = {
    "LN-A": "LINE_A", "LINE-A": "LINE_A", "L_A": "LINE_A",
    "LN-B": "LINE_B", "LINE-B": "LINE_B", "L_B": "LINE_B",
    "LN-C": "LINE_C", "LINE-C": "LINE_C", "L_C": "LINE_C",
    "LN-D": "LINE_D", "LINE-D": "LINE_D", "L_D": "LINE_D",
}

def clean_line_code(val):
    if pd.isna(val):                              # Guard against missing/NaN line codes
        return "UNKNOWN"
    s = str(val).strip().upper()                  # Remove trailing spaces ('Line_A ') and standardize case ('line_c')
    # dict.get(key, default) takes 2 parameters:
    #   1st arg (s): the key to look up in LINE_ALIASES (e.g. 'LN-B' -> 'LINE_B')
    #   2nd arg (s): the fallback value returned if the key is NOT in LINE_ALIASES (keeps 'LINE_A' or 'LINE_Z' as-is)
    return LINE_ALIASES.get(s, s)

def standardize_stoppage_type(val):
    s = str(val).strip().lower()                  # Strip spaces and lowercase for case-insensitive matching
    if "prevent" in s or "pm" in s:               # Catch 'Preventive', 'preventive', 'PM', etc.
        return "Preventive"
    if "correct" in s:                            # Catch 'Corrective', 'corrective_action', etc.
        return "Corrective"
    return "Unplanned"                            # Default category for 'unplanned', 'Unplanned', etc.

# Inspect raw values BEFORE cleaning
print("Raw telemetry IDs :", sorted(df_telem["line_id"].dropna().unique()))
print("Raw stoppage types:", df_stop["stoppage_type"].value_counts().to_dict())

# TODO: apply clean_line_code and standardize_stoppage_type to df_lines, df_telem, df_stop
#   - df_lines["line_id"]         <- clean_line_code
#   - df_telem["line_id"]         <- clean_line_code
#   - df_stop["line_code"]        <- clean_line_code
#   - df_stop["stoppage_type"]    <- standardize_stoppage_type


# Inspect cleaned values AFTER your .apply(...) calls
print("Clean telemetry IDs :", sorted(df_telem["line_id"].unique()))
print("Clean stoppage types:", df_stop["stoppage_type"].value_counts().to_dict())


## ⏱️ 6. Data Types, Timestamps, Deduplication & Night-Shift `S3` Rollover
### 6.1 Why Data Types Matter in Silver Parquet (`string` vs `timestamp` / `date`)
In Lab 02, `bronze_telemetry.parquet` intentionally preserved `timestamp` and `_ingestion_timestamp_utc` as raw **`string` (`object`)** text. Before parsing them into true Silver types, run three concrete experiments proving **why leaving timestamps/dates/numbers as strings in Silver breaks pipelines**:
1. **Broken Temporal Math & Methods:** `timestamp - pd.Timedelta(hours=6)` throws `TypeError`, `.dt.hour` throws `AttributeError`, and Lab 04 `.interpolate(method='time')` fails.
2. **Silent Lexicographical Ordering Traps:** String comparison compares ASCII characters left-to-right, NOT chronology (`'09/10/2026' > '01/11/2026'` is `True`; `'2026-10-01T06:00:00' > '2026-10-01 07:00:00'` is `True` because `'T' > ' '`!) — corrupting SQL filters and Parquet footer `min/max` pruning!
3. **RAM & Engine Contract:** Native `datetime64[ns]` uses `8 bytes/row` instead of Python string object pointers (**~8.5× less RAM**), and exposes native `TIMESTAMP` / `DATE` columns directly to DuckDB (`INTERVAL '2 HOURS'` in Lab 05) and BI tools!

In [ ]:
# Experiment 1: Temporal arithmetic fails on raw string timestamps
try:
    _ = df_telem["timestamp"].head(3) - pd.Timedelta(hours=6)
except TypeError as err:
    print(f"[Experiment 1] String timestamp arithmetic FAILED as expected -> TypeError: {err}")

# Experiment 2: Silent alphabetical vs chronological ordering bugs
eu_cmp_str = "09/10/2026" > "01/11/2026"
iso_cmp_str = "2026-10-01T06:00:00" > "2026-10-01 07:00:00"  # 'T' (ASCII 84) > ' ' (ASCII 32)!
num_cmp_str = "9" > "10"
print(f"[Experiment 2] String traps -> '09/10/2026' > '01/11/2026' is {eu_cmp_str}, '2026-10-01T06:00:00' > '2026-10-01 07:00:00' is {iso_cmp_str}, '9' > '10' is {num_cmp_str}!")

# TODO: Measure RAM usage of df_telem['timestamp'] as string (object) vs parsed datetime64[ns]
ram_str_bytes = None  # REPLACE WITH: df_telem["timestamp"].memory_usage(deep=True)
ram_dt_bytes = None   # REPLACE WITH: pd.to_datetime(df_telem["timestamp"], errors="coerce").memory_usage(deep=True)

if ram_str_bytes is not None and ram_dt_bytes is not None:
    print(f"[Experiment 3] RAM for 'timestamp' -> object/str: {ram_str_bytes:,} B vs datetime64[ns]: {ram_dt_bytes:,} B ({ram_str_bytes / ram_dt_bytes:.2f}x smaller!)")
else:
    print("⏳ [TODO] Compute ram_str_bytes and ram_dt_bytes above.")


### 6.2 Timestamp Parsing, Post-Normalization Deduplication & Night-Shift `S3` Rollover
1. Parse `df_telem['timestamp']` and `df_stop['event_date']` into native **`datetime64[ns]`** (`pd.to_datetime(..., errors='coerce')`).
2. Also convert `_ingestion_timestamp_utc` into true **`datetime64[ns, UTC]`** across all 5 tables (fulfilling Lab 02's promise!).
3. Deduplicate telemetry & stoppage logs **after** key normalization (`keep='last'`).
4. Assign `shift_code` and **`operational_shift_date = (ts - 6h).dt.date`** — kept as native **`datetime.date`** (Parquet `DATE` / `date32[day]`, **not** `.astype(str)`!) so `00:00–05:45` maps cleanly to the shift's starting operational date.

In [ ]:
# TODO:
# 1. Parse df_telem['timestamp'] and df_stop['event_date'] with pd.to_datetime(..., errors='coerce')
# 2. Deduplicate df_telem (on ["timestamp", "line_id", "bottle_format"]) and df_stop (on ["log_id"]) with keep='last' and .copy()
#    (Note: keep='last' follows current file row order top-to-bottom; .copy() prevents SettingWithCopyWarning)
# 3. Convert _ingestion_timestamp_utc to UTC datetime64[ns, UTC] on all 5 DataFrames (format="ISO8601", utc=True, errors="coerce")
# 4. Assign shift_code ('S1' 06-14, 'S2' 14-22, 'S3' 22-06) and operational_shift_date = (ts - pd.Timedelta(hours=6)).dt.date (NO .astype(str)!)

if pd.api.types.is_datetime64_any_dtype(df_telem["timestamp"]):
    print("Telemetry timestamp dtype:", df_telem["timestamp"].dtype)
else:
    print("⏳ [TODO] Parse timestamps, deduplicate, and compute shift_code + operational_shift_date above.")


## 💾 7. Enforce Silver Schema Contract, Persist `silver_part1/*.parquet` & Audit Types
Before saving to Silver Part 1, enforce an explicit **Silver Schema Contract** across all 5 tables:
- Text identifier/label columns -> **`string`** (no ambiguous `object` text columns!)
- Numeric measurements & counts -> explicit **`float64`** / **`int64`**
- Operational timestamps (`timestamp`, `event_date`) -> **`timestamp[ns]`** (`INT64` in Parquet)
- Audit lineage (`_ingestion_timestamp_utc`) -> **`timestamp[ns, tz=UTC]`**
- Business date (`operational_shift_date`) -> **`date32[day]`** (`DATE`, `INT32` in Parquet)

After writing the Parquet files, compare **Bronze vs Silver Part 1 Parquet schemas**, inspect DuckDB `DESCRIBE`, and verify that Parquet footer statistics now store native **`Timestamp`** and **`Date`** bounds!

In [ ]:
# Explicit Silver Schema Contract for all 5 tables (timestamps & dates are typed separately in Section 6.2)
SILVER_SCHEMAS = {
    "telemetry": {
        "string": ["line_id", "bottle_format", "shift_code", "_source_file"],
        "float64": ["ambient_temp_C", "pasteurizer_temp_C", "filler_pressure_bar"],
        "int64": ["conveyor_speed_bpm", "valve_cycles_k", "bottled_units", "rejected_units"],
    },
    "stoppages": {
        "string": ["log_id", "line_code", "stoppage_type", "root_cause", "technician_team", "shift_code", "_source_file"],
        "int64": ["downtime_min"],
    },
    "lines": {
        "string": ["line_id", "line_name", "filler_type", "_source_file"],
        "float64": ["nominal_cycle_time_sec", "hourly_operating_cost_eur"],
    },
    "shifts": {
        "string": ["shift_id", "shift_name", "start_time", "end_time", "_source_file"],
        "int64": ["planned_duration_hours"],
    },
    "bottle_formats": {
        "string": ["format_code", "format_name", "packaging_material", "_source_file"],
        "float64": ["unit_reject_cost_eur"],
        "int64": ["target_hourly_output"],
    },
}

def enforce_dtypes(df: pd.DataFrame, spec: dict) -> pd.DataFrame:
    out = df.copy()  # Work on an independent copy so we don't mutate the caller's DataFrame in-place
    for col in spec.get("string", []):
        out[col] = out[col].astype("string")                                  # Pandas nullable StringDtype (no ambiguous object)
    for col in spec.get("float64", []):
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64") # Coerce invalid text -> NaN, then 64-bit float
    for col in spec.get("int64", []):
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("int64")   # Cast integer counters/IDs to 64-bit int
    return out

# 1. Apply the Silver Schema Contract to all 5 DataFrames
df_telem = enforce_dtypes(df_telem, SILVER_SCHEMAS["telemetry"])
df_stop = enforce_dtypes(df_stop, SILVER_SCHEMAS["stoppages"])
df_lines = enforce_dtypes(df_lines, SILVER_SCHEMAS["lines"])
df_shifts = enforce_dtypes(df_shifts, SILVER_SCHEMAS["shifts"])
df_formats = enforce_dtypes(df_formats, SILVER_SCHEMAS["bottle_formats"])

# 2. Persist all 5 typed tables to silver_part1/*.parquet (index=False avoids writing __index_level_0__)
outputs = {
    "silver_part1_telemetry.parquet": df_telem,
    "silver_part1_stoppages.parquet": df_stop,
    "silver_lines.parquet": df_lines,
    "silver_shifts.parquet": df_shifts,
    "silver_bottle_formats.parquet": df_formats,
}
for fname, df_out in outputs.items():
    df_out.to_parquet(SILVER_P1 / fname, index=False, compression="snappy")

# 3. Verify string hygiene & categorical standardization before auditing schemas
assert df_telem["line_id"].str.contains(" ").sum() == 0, "Whitespace remains in line_id!"
assert set(df_stop["stoppage_type"]).issubset({"Unplanned", "Preventive", "Corrective"}), "Unstandardized stoppage types!"

# 4. Compare Bronze vs Silver Part 1 Arrow schemas & Parquet footer min/max statistics:
#    - Arrow schema (schema_arrow): high-level logical types used in memory by Pandas/PyArrow/DuckDB
#      (e.g. string, timestamp[ns], timestamp[ns, tz=UTC], date32[day]), preserved in file metadata.
#    - Parquet physical type (physical_type): low-level on-disk binary primitive inside Column Chunks
#      (e.g. timestamp[ns] -> INT64 on disk, date32[day] -> INT32 on disk, string -> BYTE_ARRAY).
pf_bronze = pq.ParquetFile(BRONZE_STAGED / "bronze_telemetry.parquet")
pf_silver = pq.ParquetFile(SILVER_P1 / "silver_part1_telemetry.parquet")
bronze_types = {f.name: str(f.type) for f in pf_bronze.schema_arrow}
silver_types = {f.name: str(f.type) for f in pf_silver.schema_arrow}
rg_s = pf_silver.metadata.row_group(0)
silver_stats = {rg_s.column(i).path_in_schema: rg_s.column(i) for i in range(rg_s.num_columns)}

comparison = pd.DataFrame([
    {
        "column": col,
        "bronze_arrow_type": bronze_types.get(col, "<new>"),
        "silver_arrow_type": silver_types[col],
        "parquet_phys_type": silver_stats[col].physical_type,
        "footer_min": repr(silver_stats[col].statistics.min) if silver_stats[col].statistics else None,
        "footer_max": repr(silver_stats[col].statistics.max) if silver_stats[col].statistics else None,
    }
    for col in silver_types
])
print("=== BRONZE vs SILVER PART 1 PARQUET SCHEMA & FOOTER STATS ===")
print(comparison.to_string(index=False))

# 5. Assert strict Arrow/Parquet temporal & date contracts across ALL 5 saved files
for fname in outputs:
    sc = pq.ParquetFile(SILVER_P1 / fname).schema_arrow
    names = set(sc.names)
    assert "__index_level_0__" not in names, f"Index leak in {fname}!"
    ing_t = sc.field("_ingestion_timestamp_utc").type
    assert pa.types.is_timestamp(ing_t) and ing_t.tz == "UTC", f"{fname}: _ingestion_timestamp_utc must be UTC timestamp, got {ing_t}"

assert pa.types.is_timestamp(pf_silver.schema_arrow.field("timestamp").type)
assert pa.types.is_date(pf_silver.schema_arrow.field("operational_shift_date").type)
stop_sc = pq.ParquetFile(SILVER_P1 / "silver_part1_stoppages.parquet").schema_arrow
assert pa.types.is_timestamp(stop_sc.field("event_date").type)
assert pa.types.is_date(stop_sc.field("operational_shift_date").type)

# 6. Verify DuckDB sees native TIMESTAMP / TIMESTAMPTZ / DATE types directly from Parquet
con = duckdb.connect()
duck_schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{SILVER_P1 / 'silver_part1_telemetry.parquet'}')").df()
con.close()
print("\nDuckDB DESCRIBE on silver_part1_telemetry.parquet (native TIMESTAMP / TIMESTAMPTZ / DATE!):")
print(duck_schema[["column_name", "column_type"]].to_string(index=False))

print(f"\nSaved & schema-verified all 5 Silver Part 1 Parquets in: {SILVER_P1}")
print("NOTE: LINE_Z, 2099 future dates, and -99/85 bar filler pressures are intentionally kept here for Lab 04 Quarantine!")
